# KV-Hadamard — cloud / big-GPU validation

**Author: Aryan Gupta** · repo: github.com/aryxnsdfs/kv-hadamard

Runs the tests a 12GB local GPU could not: (A) the **fp16 7B reference** that never fit locally, (B) **long-context KV-memory scaling to 32k** where the compression win becomes dramatic, (C) re-confirm **B2 packed 2-bit** quality/memory/speed at scale.

**Before running:** Runtime → Change runtime type → **A100 GPU** (High-RAM). fp16 7B needs ~24GB+; 32k fp16 KV needs an 80GB A100 (the point is it OOMs on smaller — that IS the result).

Replicates the exact pinned install from `setup_phase2.sh` (transformers 4.36.2, flash-attn 2.5.6, gcc-12 for the KIVI CUDA kernels). Total setup ~15-30 min (flash-attn / kernel build).


## 0. GPU check — confirm you got an A100

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nvcc --version | tail -1 || echo 'nvcc missing (Colab usually has it)'
import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, torch.cuda.is_available())

## 1. System deps — gcc-12 (CUDA 12.x nvcc rejects gcc>12)

In [ ]:
!sudo apt-get update -qq && sudo apt-get install -y -qq gcc-12 g++-12
import os
os.environ['CC']='gcc-12'; os.environ['CXX']='g++-12'; os.environ['CUDAHOSTCXX']='g++-12'
print('gcc-12 set for the CUDA kernel builds')

## 2. Clone KIVI + this repo

In [ ]:
%cd /content
![ -d KIVI ] || git clone -q https://github.com/jy-yuan/KIVI.git
![ -d kv-hadamard ] || git clone -q https://github.com/aryxnsdfs/kv-hadamard.git
print('cloned KIVI + kv-hadamard')

## 3. Python deps — the pinned combo from setup_phase2.sh

Colab already has a CUDA-12 torch; we keep it and pin the transformers/bnb/accelerate KIVI was validated against.

In [ ]:
!pip install -q ninja packaging
!pip install -q transformers==4.36.2 accelerate==0.25.0 bitsandbytes==0.43.0
!pip install -q datasets sentencepiece protobuf matplotlib
print('deps installed')

## 4. flash-attn 2.5.6 (KIVI asserts use_flash=True)

Source build is slow (~15 min) and RAM-heavy. Limit jobs to avoid an OOM kill. If it fails, use the prebuilt-wheel fallback in the next cell.

In [ ]:
import os
os.environ['MAX_JOBS']='4'
!pip install -q flash-attn==2.5.6 --no-build-isolation || echo 'BUILD FAILED -> run the fallback cell below'

In [ ]:
# FALLBACK (only if the build above failed): grab a prebuilt wheel.
# Match cp310/311 + torch major + cu12 + abiFALSE to your runtime. Check
# https://github.com/Dao-AILab/flash-attention/releases for the exact name.
# import sys; print(sys.version)  # confirm your python tag, then:
# !pip install -q https://github.com/Dao-AILab/flash-attention/releases/download/v2.5.6/flash_attn-2.5.6+cu122torch2.2cxx11abiFALSE-cp310-cp310-linux_x86_64.whl
print('only run this if the source build failed')

## 5. Build KIVI's CUDA quant kernels (--no-build-isolation so it sees torch)

In [ ]:
!cd /content/KIVI/quant && pip install -e . --no-build-isolation 2>&1 | tail -5
print('KIVI kernels built')

## 6. Import sanity — everything wired up?

In [ ]:
import sys
sys.path.insert(0, '/content/KIVI')
sys.path.insert(0, '/content/kv-hadamard')
%cd /content/kv-hadamard
import torch, transformers
print('transformers', transformers.__version__)
import flash_attn; print('flash-attn', flash_attn.__version__)
from models.llama_kivi import LlamaForCausalLM_KIVI; print('KIVI import OK')
from b2_packed_int2 import patch_packed_int2_v; print('B2 import OK')
from phase2_kivi_harness import load_kivi, load_fp16, CONFIG
from fp16_baseline_harness import measure_kv_memory, measure_decode_tps, free_cuda
from probe_cache_ppl import ppl_cache, PREFILL
print('ALL IMPORTS OK — ready to test')

## TEST A — fp16 7B reference (the baseline that never fit on 12GB)

Three-way on Llama-2-7B at a valid context: **fp16 (exact KV)** vs **KIVI int4** vs **B2 packed Hadamard-INT2**. Cache-path PPL + real KV memory + decode speed. This gives the headline fp16 reference the local runs were missing.

In [ ]:
import math, torch, torch.nn.functional as F
from datasets import load_dataset

def passage(tok, N, offset=0):
    ds = load_dataset(CONFIG['wikitext_dataset'], CONFIG['wikitext_config'], split=CONFIG['wikitext_split'])
    ids = tok('\n\n'.join(ds['text']), return_tensors='pt').input_ids
    return ids[:, offset:offset+N]

CONFIG['model_name'] = 'NousResearch/Llama-2-7b-hf'
results_A = {}

# --- fp16 (now fits on A100) ---
print('=== fp16 reference ===', flush=True)
model, tok, _ = load_fp16()
ids = passage(tok, 896).to(CONFIG['device'])
results_A['fp16'] = {'ppl': ppl_cache(model, ids), 'kv_mb': measure_kv_memory(model, tok, 2048)['empirical_mb'], 'tps': measure_decode_tps(model, tok)['tokens_per_sec']}
print('fp16', results_A['fp16'], flush=True)
del model; free_cuda()

In [ ]:
# --- kivi + B2 (same load, patch for B2) ---
print('=== kivi ===', flush=True)
model, tok, _ = load_kivi()
ids = passage(tok, 896).to(CONFIG['device'])
results_A['kivi'] = {'ppl': ppl_cache(model, ids), 'kv_mb': measure_kv_memory(model, tok, 2048)['empirical_mb'], 'tps': measure_decode_tps(model, tok)['tokens_per_sec']}
print('kivi', results_A['kivi'], flush=True); free_cuda()

print('=== B2 packed Hadamard-INT2 ===', flush=True)
patch_packed_int2_v(model)
results_A['b2'] = {'ppl': ppl_cache(model, ids), 'kv_mb': measure_kv_memory(model, tok, 2048)['empirical_mb'], 'tps': measure_decode_tps(model, tok)['tokens_per_sec']}
print('b2', results_A['b2'], flush=True)
del model; free_cuda()

print('\n=== TEST A SUMMARY (Llama-2-7B) ===')
print(f"{'':<8}{'PPL':>10}{'KV MB@2048':>14}{'tok/s':>10}")
for k in ['fp16','kivi','b2']:
    r=results_A[k]; print(f"{k:<8}{r['ppl']:>10.4f}{r['kv_mb']:>14.1f}{r['tps']:>10.1f}")

## TEST B — long-context KV-memory scaling to 32k (the dramatic win)

Uses `togethercomputer/LLaMA-2-7B-32K` — **Llama-2 architecture** (extended RoPE) so KIVI's class loads it, but a 32k context window. We measure **real KV-cache bytes** at 4k / 8k / 16k / 32k for fp16 vs KIVI vs B2. Expectation: fp16 KV OOMs at long context on smaller cards; B2 stays small. Memory only (fast prefill) — quality already proven at short context in Test A.

In [ ]:
CONFIG['model_name'] = 'togethercomputer/LLaMA-2-7B-32K'
LENGTHS = [4096, 8192, 16384, 32768]
results_B = {'lengths': LENGTHS}

def kv_scan(tag, loader, patch=None):
    print(f'=== {tag} KV scan ===', flush=True)
    model, tok, _ = loader()
    if patch: patch(model)
    row = {}
    for N in LENGTHS:
        try:
            mb = measure_kv_memory(model, tok, N)['empirical_mb']
            row[N] = round(mb, 1); print(f'  {tag} @{N}: {mb:.1f} MB', flush=True)
        except Exception as e:
            row[N] = 'OOM' if ('out of memory' in str(e).lower() or 'CUDA' in str(e)) else f'ERR:{e}'
            print(f'  {tag} @{N}: {row[N]}', flush=True); free_cuda()
    del model; free_cuda(); return row

results_B['kivi'] = kv_scan('kivi', load_kivi)
results_B['b2']   = kv_scan('b2',   load_kivi, patch=patch_packed_int2_v)
# fp16 last — most likely to OOM at 32k (that OOM is a headline result)
results_B['fp16'] = kv_scan('fp16', load_fp16)
print('\nTEST B done:', results_B)

## Plots + save

In [ ]:
import json, matplotlib.pyplot as plt, numpy as np
json.dump({'A': results_A, 'B': results_B}, open('cloud_results.json','w'), indent=2)

# Test A bars
fig, axs = plt.subplots(1, 3, figsize=(13,4))
ks=['fp16','kivi','b2']; cols=['#8a8a8a','#444','#1a7f3c']
for ax, metric, ttl in zip(axs, ['ppl','kv_mb','tps'], ['Cache-path PPL','KV MB @2048','decode tok/s']):
    ax.bar(ks, [results_A[k][metric] for k in ks], color=cols, edgecolor='k')
    ax.set_title(ttl)
fig.suptitle('Test A — Llama-2-7B: fp16 vs KIVI vs B2 packed Hadamard-INT2'); fig.tight_layout()
fig.savefig('cloud_testA.png'); plt.show()

# Test B memory scaling (numeric only)
fig, ax = plt.subplots(figsize=(8,5))
for k,c in [('fp16','#8a8a8a'),('kivi','#444'),('b2','#1a7f3c')]:
    xs=[N for N in LENGTHS if isinstance(results_B[k].get(N),(int,float))]
    ys=[results_B[k][N] for N in xs]
    if xs: ax.plot([x//1024 for x in xs], ys, 'o-', label=k, color=c, lw=2)
ax.set_xlabel('context length (k tokens)'); ax.set_ylabel('KV cache MB (measured)')
ax.set_title('Test B — KV memory scaling (missing points = OOM)'); ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig('cloud_testB.png'); plt.show()

from google.colab import files
files.download('cloud_results.json'); files.download('cloud_testA.png'); files.download('cloud_testB.png')